In [ ]:
import os
import json
import time
import requests
from requests.auth import HTTPBasicAuth

from eodag import EODataAccessGateway


Define variables:

In [38]:
GRASS_PROJECT = "s2_proj_epsg32635"
START = '2025-12-01'
END = '2025-12-03'
TILE_ID = "35SKC"
MAIN_PROCESS_CHAIN = "pc_MAIN.json"


Define actinia variables:

In [ ]:
# variables to set the actinia host, version, and user
actinia_baseurl = "http://localhost:8088"
actinia_version = "v3"
actinia_url = actinia_baseurl + "/api/" + actinia_version
actinia_auth = HTTPBasicAuth('actinia', 'actinia')  # username, password


### Filter Sentinel-2 scenes

In [39]:
search_criteria = {
    "productType": "S2MSI2A",
    "start": START,
    "end": END,
    "tileIdentifier": TILE_ID,
}


In [40]:
dag = EODataAccessGateway()
all_products = dag.search_all(**search_criteria)
s2_ids = [all_products[i].properties['id']  for i in range(len(all_products))]
print(f"Found <{len(s2_ids)}> Sentinel-2 scene IDs:")
for id in s2_ids:
    print(f" - {id}")


Found <1> Sentinel-2 scene IDs:
 - S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209


### Start actinia

In [6]:
# helper function to print formatted JSON using the json module

def print_as_json(data):
    print(json.dumps(data, indent=2))

# helper function to verify a request
def verify_request(request, success_code=200):
    if request.status_code != success_code:
        print("ERROR: actinia processing failed with status code %d!" % request.status_code)
        print("See errors below:")
        print_as_json(request.json())
        request_url = request.json()["urls"]["status"]
        requests.delete(url=request_url, auth=actinia_auth)
        raise Exception("The resource <%s> has been terminated." % request_url)

Construct actinia endpoint:
`...locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing` -> persistent, data is kept, needs
`...locations/{GRASS_PROJECT}/processing_export` -> non-persistent, data is not kept in GRASSDB

In [42]:
# make a POST request to the actinia data API
request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/processing_export"
# request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing_export"

print("actinia POST request:")
print(request_url)
print("---")

actinia POST request:
http://localhost:8088/api/v3/locations/s2_proj_epsg32635/processing_export
---


In [47]:
# loop over all Sentinel-2 scene IDs and start processing
# list for storing status request URLs
request_url_lst = []
for s2_id in s2_ids:
    print(f"Processing Sentinel-2 scene ID: {s2_id}")
    # load the main process chain
    process_chain = json.load(open(MAIN_PROCESS_CHAIN))
    # modify the process chain with the current Sentinel-2 scene ID
    process_chain["list"][0]["inputs"][0]["value"] = s2_id
    # modify the output names for NDVI and NDWI
    process_chain["list"][5]["inputs"][2]['value'] = f"NDVI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"
    process_chain["list"][6]["inputs"][2]['value'] = f"NDWI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"

    # make the POST request to start the processing
    request = requests.post(url=request_url, auth=actinia_auth, json=process_chain)
    # check if anything went wrong
    verify_request(request, 200)
    # get a json-encoded content of the response
    jsonResponse = request.json()
    # save the status request URL
    request_url_lst.append(jsonResponse["urls"]["status"])

    while jsonResponse["status"] in ("accepted", "running"):
        status_request_url = jsonResponse["urls"]["status"]
        status_request = requests.get(url=status_request_url.replace("https", "http"), auth=actinia_auth)
        jsonResponse = status_request.json()
        print(f"Polling status for {s2_id}: {jsonResponse['status']}")
        print(f"Doing: {jsonResponse['message']}")
        print('#########################################')
        time.sleep(30)

    print(f"Final status for {s2_id}: {jsonResponse['status']}")
    print(f"Processing for {s2_id} finished.")
    print('=========================================')




Processing Sentinel-2 scene ID: S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209
Polling status for S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209: accepted
Doing: Resource accepted
#########################################
Polling status for S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209: running
Doing: Running executable i.s2_id.download with parameters ['s2_id=S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'download_dir=/home/data/'] for 30.133605480194092 seconds
#########################################
Polling status for S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209: running
Doing: Running executable i.s2_id.download with parameters ['s2_id=S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'download_dir=/home/data/'] for 60.2715699672699 seconds
#########################################
Polling status for S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209: running
Doing: Running executab

In [46]:
jsonResponse

{'accept_datetime': '2025-12-11 13:01:44.466242',
 'accept_timestamp': 1765458104.46624,
 'api_info': {'endpoint': 'gdiasyncephemeralexportresource',
  'method': 'POST',
  'path': '/api/v3/locations/s2_proj_epsg32635/processing_export',
  'request_url': 'http://localhost:8088/api/v3/locations/s2_proj_epsg32635/processing_export'},
 'datetime': '2025-12-11 13:05:49.946355',
 'exception': {'message': 'AsyncProcessError:  Error while running executable <r.mask>',
  'traceback': ['  File "/opt/venv/lib/python3.12/site-packages/actinia_processing_lib/ephemeral_processing.py", line 2179, in run\n    self._execute()\n',
   '  File "/opt/venv/lib/python3.12/site-packages/actinia_processing_lib/ephemeral_processing_with_export.py", line 545, in _execute\n    EphemeralProcessing._execute(self)\n',
   '  File "/opt/venv/lib/python3.12/site-packages/actinia_processing_lib/ephemeral_processing.py", line 1937, in _execute\n    self._execute_process_list(process_list=process_list)\n',
   '  File "/op

Print links to tif of NDVI/NDWI results

In [48]:
# print links to processed tifs
for s2_id in request_url_lst:
    status_request = requests.get(url=s2_id.replace("https", "http"), auth=actinia_auth)
    jsonResponse = status_request.json()
    for tif in jsonResponse['urls']['resources']:
        print(f"{os.path.basename(tif)}: {tif.replace('https', 'http')}")

NDVI_20251202_S2A.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-5ba58534-343c-4b49-b99f-4d11b80b04bf/NDVI_20251202_S2A.tif
NDWI_20251202_S2A.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-5ba58534-343c-4b49-b99f-4d11b80b04bf/NDWI_20251202_S2A.tif
